# Sacred -> Techno v2 backend (Colab Pro)

LoRA 파인튜닝 모델 + 보컬 chop + 드럼 합성 + 사이드체인 믹스 방식의 v2 파이프라인을 실행합니다.
프론트엔드 UI는 그대로이고, `/convert` 엔드포인트 계약도 동일합니다.

## 사전 설정 (한 번만)

1. https://dashboard.ngrok.com 에 가입하고 Auth Token을 확인합니다.
2. https://dashboard.ngrok.com/domains 에서 고정 도메인(static domain)을 발급받습니다.
3. Colab 왼쪽 사이드바 열쇠 아이콘(Secrets)에 다음을 등록하고 'Notebook access'를 켜주세요.
   - `NGROK_AUTH_TOKEN`: ngrok 토큰
   - `NGROK_DOMAIN`: 고정 도메인 (`https://` 제외, 예: `your-name.ngrok-free.app`)
4. Vercel `NEXT_PUBLIC_ML_BACKEND_URL`을 `https://<고정 도메인>`으로 설정하고 재배포합니다 (한 번만).

## 발표 직전에 할 일

1. 런타임 → 런타임 유형 변경 → GPU (L4 권장)
2. 아래 셀을 순서대로 실행
   - **셀 5 (모델 다운로드)**: 처음 실행 시 MusicGen (~1.5 GB) + LoRA 어댑터 다운로드로 수 분 소요. 진행 상황이 출력됩니다. 두 번째 실행부터는 캐시에서 로드되어 빠릅니다.
   - **셀 6 (서버 기동)**: 모델이 캐시되어 있으므로 30초 내외로 완료됩니다.
3. 마지막 셀에 'Backend URL'이 출력되고 고정 도메인과 일치하면 완료
4. 노트북을 끄지 말고 발표가 끝날 때까지 켜둔 상태로 유지


In [ ]:
import os

if not os.path.exists("/content/MIR"):
    !git clone https://github.com/rocgwang/MIR.git /content/MIR
else:
    !cd /content/MIR && git pull

%cd /content/MIR/backend_v2

In [ ]:
!pip install -q -r requirements.txt
!pip install -q pyngrok
!pip install -q "torchao>=0.16.0"  # Colab 기본 설치(0.10.0)는 PEFT와 충돌


In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
from google.colab import userdata
from pyngrok import ngrok

NGROK_AUTH_TOKEN = userdata.get("NGROK_AUTH_TOKEN")
NGROK_DOMAIN = userdata.get("NGROK_DOMAIN")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

In [ ]:
import os
import gdown
from transformers import AutoProcessor, MusicgenForConditionalGeneration

# LoRA 어댑터 다운로드 (없을 때만)
LORA_PATH = "/tmp/lora_adapters"
if not os.path.exists(LORA_PATH):
    print("LoRA 어댑터 다운로드 중...")
    GDRIVE_URL = os.getenv(
        "LORA_ADAPTER_GDRIVE_URL",
        "https://drive.google.com/drive/folders/1cxKcfxQB7GTl4IEX71e09wVE0oTXPwCU?usp=sharing",
    )
    gdown.download_folder(GDRIVE_URL, output=LORA_PATH, quiet=False)
    print("LoRA 다운로드 완료 ✓")
else:
    print("LoRA 어댑터 캐시 있음, 건너뜀 ✓")

# MusicGen-Medium 가중치 다운로드 (~1.5 GB, 처음 한 번만 HuggingFace 캐시에 저장)
print("\nMusicGen-Medium 모델 다운로드 중 (처음 실행 시 수 분 소요)...")
AutoProcessor.from_pretrained("facebook/musicgen-medium")
MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-medium")
print("MusicGen 다운로드 완료 ✓")


In [ ]:
import subprocess
import time
import urllib.request

LOG_FILE = "/tmp/uvicorn.log"
log_f = open(LOG_FILE, "w")

server = subprocess.Popen(
    ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd="/content/MIR/backend_v2",
    stdout=log_f,
    stderr=log_f,
)

print("서버 기동 중 (모델이 캐시되어 있으므로 30초 내외)...")
start = time.time()
TIMEOUT_S = 300

while True:
    if server.poll() is not None:
        log_f.flush()
        print(f"\n❌ 서버가 예기치 않게 종료됐습니다 (종료 코드: {server.poll()})")
        print("\n--- 서버 로그 (오류 내용) ---")
        with open(LOG_FILE) as f:
            print(f.read())
        break
    elapsed = time.time() - start
    if elapsed > TIMEOUT_S:
        log_f.flush()
        print(f"\n⚠️  5분 내에 서버가 응답하지 않았습니다.")
        print("\n--- 서버 로그 ---")
        with open(LOG_FILE) as f:
            print(f.read())
        break
    try:
        urllib.request.urlopen("http://localhost:8000/openapi.json", timeout=2)
        log_f.close()
        print(f"\n✅ 서버 준비 완료 ({elapsed:.0f}초)")
        break
    except Exception:
        print(f"  대기 중... {elapsed:.0f}초 경과", end="\r")
        time.sleep(3)


In [ ]:
tunnel = ngrok.connect(8000, domain=NGROK_DOMAIN)
print("Backend URL:", tunnel.public_url)
print("이 URL이 Vercel의 NEXT_PUBLIC_ML_BACKEND_URL 값과 같은지 확인하세요.")

## 발표 종료 후

발표가 끝나면 '런타임 > 런타임 연결 해제 및 삭제'로 종료해서 Colab Pro 사용량을 절약하세요.